# Construct match-level Elo ratings

This notebook creates leakage-free **pre-match Elo ratings** from `results.csv`.

Key choices:

- all `Olympic Games` matches are excluded because Olympic squads are not generally senior first-choice national teams;
- Elo uses the reported score, which includes extra time but excludes penalty-shootout kicks;
- all teams start at 1,500 and ratings with fewer than 30 prior matches are flagged as provisional;
- genuine home teams receive a 100-point home advantage;
- when a team appears more than once on the same date, every match uses its start-of-day rating and the combined changes are carried forward;
- the output variables used later for modelling are explicitly named `home_elo_pre` and `away_elo_pre`.


In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    """Locate the repo when Jupyter starts in its root or a subfolder."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "01 Input").is_dir() and (candidate / "02 Scripts").is_dir():
            return candidate
    raise FileNotFoundError("Start Jupyter from the project folder or 02 Scripts")


project_root = find_project_root()

input_dir = project_root / "01 Input"
temp_dir = project_root / "03 Temp"
output_dir = project_root / "04 Output"
audit_dir = output_dir / "data_audits"

for folder in [temp_dir, output_dir, audit_dir]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


## 1. Import and clean the match results

In [ ]:
# Only results.csv is needed to construct Elo ratings.
results = pd.read_csv(
    input_dir / "results.csv",
    parse_dates=["date"],
)

print(f"Rows imported: {len(results):,}")
print(
    "Date range:",
    results["date"].min().date(),
    "to",
    results["date"].max().date(),
)

display(results.head())


In [ ]:
# Retain the source order as a deterministic tie-breaker within a date.
results = results.copy()
results["_source_order"] = np.arange(len(results))

# Remove scheduled fixtures without scores.
unplayed_matches = results.loc[
    results[["home_score", "away_score"]].isna().any(axis=1)
].copy()

matches = results.dropna(subset=["home_score", "away_score"]).copy()
matches[["home_score", "away_score"]] = matches[
    ["home_score", "away_score"]
].astype(int)

# Drop every Olympic match. These do not consistently represent senior,
# first-choice national teams and are not part of the prediction sample.
olympic_matches = matches.loc[
    matches["tournament"].eq("Olympic Games")
].copy()

matches = matches.loc[
    ~matches["tournament"].eq("Olympic Games")
].copy()

unplayed_matches.to_csv(
    audit_dir / "unplayed_matches.csv",
    index=False,
)
olympic_matches.to_csv(
    audit_dir / "excluded_olympic_matches.csv",
    index=False,
)

print(f"Unplayed fixtures removed: {len(unplayed_matches):,}")
print(f"Olympic matches removed: {len(olympic_matches):,}")


In [ ]:
# Remove duplicate rows describing the same match. City is deliberately
# excluded because the same match can occasionally appear with two city labels.
match_identity = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "country",
    "neutral",
]

before_deduplication = len(matches)

duplicate_matches = matches.loc[
    matches.duplicated(match_identity, keep=False)
].copy()

matches = (
    matches.sort_values("_source_order")
    .drop_duplicates(match_identity, keep="first")
    .copy()
)

duplicate_matches.to_csv(
    audit_dir / "duplicate_matches.csv",
    index=False,
)

print(
    "Duplicate rows removed:",
    f"{before_deduplication - len(matches):,}",
)


In [ ]:
# If the same two teams have conflicting records on the same date,
# exclude all versions rather than choosing a score arbitrarily.
match_key = ["date", "home_team", "away_team"]

conflicting_keys = (
    matches.groupby(match_key)
    .size()
    .loc[lambda count: count > 1]
    .reset_index()[match_key]
)

conflicting_matches = matches.merge(
    conflicting_keys,
    on=match_key,
    how="inner",
)

matches = (
    matches.merge(
        conflicting_keys.assign(conflicting_record=True),
        on=match_key,
        how="left",
    )
    .loc[lambda data: data["conflicting_record"].isna()]
    .drop(columns="conflicting_record")
)

conflicting_matches.to_csv(
    audit_dir / "conflicting_match_records.csv",
    index=False,
)

matches = matches.sort_values(["date", "_source_order"]).reset_index(drop=True)
matches["match_id"] = np.arange(1, len(matches) + 1)

print(f"Conflicting rows removed: {len(conflicting_matches):,}")
print(f"Usable completed matches: {len(matches):,}")

display(conflicting_matches)


In [ ]:
# Audit countries appearing more than once on a date. These observations
# are retained and handled with start-of-day Elo ratings below.
team_dates = pd.concat(
    [
        matches[["match_id", "date", "home_team"]].rename(
            columns={"home_team": "team"}
        ),
        matches[["match_id", "date", "away_team"]].rename(
            columns={"away_team": "team"}
        ),
    ],
    ignore_index=True,
)

multiple_matches_same_day = (
    team_dates.groupby(["date", "team"], as_index=False)
    .agg(
        matches_on_date=("match_id", "size"),
        match_ids=("match_id", lambda values: ",".join(values.astype(str))),
    )
    .loc[lambda data: data["matches_on_date"] > 1]
)

multiple_matches_same_day.to_csv(
    audit_dir / "teams_with_multiple_matches_same_day.csv",
    index=False,
)

print(
    "Team-date combinations with multiple matches:",
    len(multiple_matches_same_day),
)

display(multiple_matches_same_day.head(30))


## 2. Assign match-importance weights

The World Football Elo framework uses larger updates for more important matches. The mapping is explicit so that it can be reviewed and amended rather than hidden inside the rating function.


In [ ]:
# Major continental finals and intercontinental senior competitions.
continental_finals = {
    "AFC Asian Cup",
    "African Cup of Nations",
    "Copa América",
    "CONCACAF Championship",
    "Confederations Cup",
    "Gold Cup",
    "Oceania Nations Cup",
    "UEFA Euro",
    "CONMEBOL–UEFA Cup of Champions",
}

# Qualifiers for the World Cup or a continental championship.
major_qualifiers = {
    "FIFA World Cup qualification",
    "AFC Asian Cup qualification",
    "African Cup of Nations qualification",
    "Copa América qualification",
    "CONCACAF Championship qualification",
    "Gold Cup qualification",
    "Oceania Nations Cup qualification",
    "UEFA Euro qualification",
}

# Other senior competitions treated as major tournaments.
major_tournaments = {
    "UEFA Nations League",
    "CONCACAF Nations League",
}

k_factor_map = {
    "Friendly": 20,
    "FIFA World Cup": 60,
    **{name: 50 for name in continental_finals},
    **{name: 40 for name in major_qualifiers | major_tournaments},
}

# Other tournaments receive the standard K = 30 weight.
matches["k_factor"] = (
    matches["tournament"].map(k_factor_map).fillna(30).astype("int8")
)

k_rule_map = {
    20: "friendly",
    30: "other_tournament",
    40: "qualifier_or_major_tournament",
    50: "continental_or_intercontinental_final",
    60: "world_cup",
}

matches["k_rule"] = matches["k_factor"].map(k_rule_map)


In [ ]:
# Save the complete competition mapping for manual review.
tournament_k_map = (
    matches[["tournament", "k_factor", "k_rule"]]
    .drop_duplicates()
    .sort_values(["k_factor", "tournament"])
)

tournament_k_map.to_csv(
    output_dir / "tournament_k_map.csv",
    index=False,
)

display(tournament_k_map)


## 3. Calculate chronological pre-match Elo ratings

In [ ]:
def expected_home_score(home_elo, away_elo, neutral):
    """Expected home-team result, where a draw counts as 0.5."""

    home_advantage = 0 if neutral else 100
    rating_difference = home_elo - away_elo + home_advantage
    return 1 / (1 + 10 ** (-rating_difference / 400))


def goal_multiplier(goal_difference):
    """World Football Elo adjustment for the winning margin."""

    if goal_difference <= 1:
        return 1.0
    if goal_difference == 2:
        return 1.5
    return (11 + goal_difference) / 8


def round_rating_change(value):
    """Round to the nearest integer, with halves away from zero."""

    return (
        int(np.floor(value + 0.5))
        if value >= 0
        else int(np.ceil(value - 0.5))
    )


In [ ]:
def build_elo_history(matches, initial_elo=1500, provisional_matches=30):
    """Return match-level pre-match Elo ratings and the latest ratings."""

    matches = matches.sort_values(["date", "_source_order"]).copy()
    ratings = defaultdict(lambda: float(initial_elo))
    matches_played = defaultdict(int)
    elo_rows = []

    # Kickoff times are unavailable. All matches on the same date therefore
    # use start-of-day ratings, and their Elo changes are applied together.
    for _, daily_matches in matches.groupby("date", sort=True):
        daily_changes = defaultdict(int)
        daily_appearances = defaultdict(int)

        for match in daily_matches.itertuples(index=False):
            home = match.home_team
            away = match.away_team
            home_elo = ratings[home]
            away_elo = ratings[away]

            expected_home = expected_home_score(
                home_elo,
                away_elo,
                match.neutral,
            )

            if match.home_score > match.away_score:
                actual_home, result = 1.0, "H"
            elif match.home_score < match.away_score:
                actual_home, result = 0.0, "A"
            else:
                actual_home, result = 0.5, "D"

            margin = abs(match.home_score - match.away_score)
            home_change = round_rating_change(
                match.k_factor
                * goal_multiplier(margin)
                * (actual_home - expected_home)
            )

            # These are the ratings available before the current match.
            elo_rows.append(
                {
                    "match_id": match.match_id,
                    "home_elo_pre": home_elo,
                    "away_elo_pre": away_elo,
                    "elo_diff_pre": home_elo - away_elo,
                    "home_expected_score": expected_home,
                    "elo_result": result,
                    "home_elo_change": home_change,
                    "away_elo_change": -home_change,
                    "home_matches_pre": matches_played[home],
                    "away_matches_pre": matches_played[away],
                    "home_provisional_pre": (
                        matches_played[home] < provisional_matches
                    ),
                    "away_provisional_pre": (
                        matches_played[away] < provisional_matches
                    ),
                }
            )

            daily_changes[home] += home_change
            daily_changes[away] -= home_change
            daily_appearances[home] += 1
            daily_appearances[away] += 1

        # Carry the combined changes into every team's next match date.
        for team, change in daily_changes.items():
            ratings[team] += change

        for team, appearances in daily_appearances.items():
            matches_played[team] += appearances

    elo_history = matches.merge(
        pd.DataFrame(elo_rows),
        on="match_id",
        how="left",
        validate="one_to_one",
    )

    current_ratings = pd.DataFrame(
        {
            "team": list(ratings),
            "elo_rating": [ratings[team] for team in ratings],
            "matches_played": [matches_played[team] for team in ratings],
        }
    )

    current_ratings["provisional"] = (
        current_ratings["matches_played"] < provisional_matches
    )

    current_ratings = (
        current_ratings.sort_values("elo_rating", ascending=False)
        .reset_index(drop=True)
    )
    current_ratings["elo_rank"] = current_ratings.index + 1

    return elo_history, current_ratings


In [ ]:
# Use the last day before the World Cup as the pre-tournament cutoff.
# Change this date only if constructing a later, live information set.
as_of_date = pd.Timestamp("2026-06-10")

elo_input = matches.loc[matches["date"].le(as_of_date)].copy()
elo_history, current_ratings = build_elo_history(elo_input)

print(f"Matches processed: {len(elo_history):,}")
print(f"Last match date: {elo_history['date'].max().date()}")
print(f"Teams rated: {len(current_ratings):,}")

display(current_ratings.head(30))


## 4. Validate and save the Elo outputs

In [ ]:
# Basic internal checks.
assert elo_history["home_elo_pre"].notna().all()
assert elo_history["away_elo_pre"].notna().all()
assert (
    elo_history["home_elo_change"] + elo_history["away_elo_change"]
).eq(0).all()
assert elo_history["home_expected_score"].between(0, 1).all()
assert elo_history["elo_result"].isin(["H", "D", "A"]).all()

# Confirm that each rating equals 1,500 plus all changes from earlier dates.
home_history = elo_history[
    ["date", "home_team", "home_elo_pre", "home_elo_change"]
].rename(
    columns={
        "home_team": "team",
        "home_elo_pre": "elo_pre",
        "home_elo_change": "elo_change",
    }
)

away_history = elo_history[
    ["date", "away_team", "away_elo_pre", "away_elo_change"]
].rename(
    columns={
        "away_team": "team",
        "away_elo_pre": "elo_pre",
        "away_elo_change": "elo_change",
    }
)

team_date_history = (
    pd.concat([home_history, away_history], ignore_index=True)
    .groupby(["team", "date"], as_index=False)
    .agg(elo_pre=("elo_pre", "first"), elo_change=("elo_change", "sum"))
    .sort_values(["team", "date"])
)

team_date_history["expected_elo_pre"] = (
    1500
    + team_date_history.groupby("team")["elo_change"].cumsum()
    - team_date_history["elo_change"]
)

assert np.allclose(
    team_date_history["elo_pre"],
    team_date_history["expected_elo_pre"],
)

print("All Elo validation checks passed.")


In [ ]:
# Save the match-level history for later feature engineering and modelling.
elo_history.to_parquet(
    temp_dir / "match_level_elo_history.parquet",
    index=False,
)

# Save the pre-tournament ratings as a convenient output table.
current_ratings.to_parquet(
    temp_dir / "current_elo_ratings.parquet",
    index=False,
)
current_ratings.to_csv(
    output_dir / "current_elo_ratings.csv",
    index=False,
)

print("Elo outputs saved.")
